# **LangChain & GenAI Essential Code Notebook**
This notebook contains practical starter code for building GenAI applications with **LangChain**.

Sections:
- Setup
- Basic LLM Call
- Prompt Templates
- Chains (LCEL)
- RAG (Retrieval Augmented Generation)
- Vector Databases (FAISS)
- Memory
- Tools & Agents
- Streaming Responses


## **LangChain** 


LangChain is a framework for building applications using large language models (LLMs). It helps developers connect LLMs with external data sources, tools, and workflows. LangChain enables features like chatbots, question-answering systems, document search (RAG), and AI agents. It simplifies prompt management, memory handling, and integration with databases, APIs, and vector stores for intelligent AI applications.

## **1. Install Required Libraries**

In [ ]:
!pip install langchain langchain-openai langchain-community faiss-cpu python-dotenv

## **2. Load API Key**

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## **3. Basic LLM Call**

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke("Explain machine learning in simple words")
print(response.content)

## **4. Prompt Templates**

Prompt Templates are reusable structures used to create prompts for large language models (LLMs). Instead of writing a new prompt every time, you define a template with placeholders (variables) that can be filled dynamically with input data.

In [ ]:
from langchain.prompts import PromptTemplate

template = "Explain {topic} in simple terms"

prompt = PromptTemplate(
    input_variables=["topic"],
    template=template
)

print(prompt.format(topic="neural networks"))

## **5. LangChain Expression Language (LCEL) Chain**

LCEL Chains (LangChain Expression Language Chains) are a way to connect different components in LangChain—such as prompts, language models, and output parsers—into a single pipeline using a simple expression syntax.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

chain.invoke({"topic": "deep learning"})

## **6. Document Loading**

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("sample.txt")
documents = loader.load()

documents[:1]

## **7. Text Splitting**

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
len(chunks)

## **8. Embeddings + FAISS Vector Store**

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

## **9. Retriever**

In [ ]:
retriever = vectorstore.as_retriever()

docs = retriever.invoke("What is machine learning?")
docs[:1]

## **10. RAG Chain**

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

qa_chain.invoke("Explain the document")

## **11. Memory (Chat History)**

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory
)

conversation.predict(input="Hello")

## **12. Tools**

In [ ]:
from langchain.tools import tool

@tool
def multiply(a:int,b:int)->int:
    "Multiply two numbers"
    return a*b

## **13. Agents**

In [ ]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType

agent = initialize_agent(
    tools=[multiply],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.run("What is 5 times 6?")